In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

from __future__ import annotations
import argparse

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import joblib
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score, recall_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from scipy.stats import ks_2samp # => Kiểm tra numerical distribution
from sklearn.compose import ColumnTransformer
from sklearn.base import clone

import os
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
df_test = pd.read_parquet("/kaggle/input/datasets/dhoogla/unswnb15/UNSW_NB15_testing-set.parquet")
df_train = pd.read_parquet("/kaggle/input/datasets/dhoogla/unswnb15/UNSW_NB15_training-set.parquet")
full_df = pd.concat([df_train, df_test], ignore_index=True)

In [3]:
df_test.sample(5)

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,attack_cat,label
59721,0.000007,udp,dns,INT,2,0,114,0,142857.140625,6.514286e+07,...,0,0,4,4,0,0,0,0,Generic,1
35167,0.000964,udp,dns,CON,2,2,146,178,3112.033203,6.058091e+05,...,0,0,2,1,0,0,0,0,Normal,0
63689,0.000004,udp,-,INT,2,0,168,0,250000.000000,1.680000e+08,...,0,0,1,1,0,0,0,0,Reconnaissance,1
67637,1.640988,tcp,ftp,FIN,14,12,756,682,15.234725,3.422329e+03,...,0,0,1,1,0,0,0,0,Normal,0
24668,0.068643,tcp,-,FIN,54,56,3302,40550,1587.925903,3.777224e+05,...,0,0,1,1,0,0,0,0,Normal,0


In [4]:
df_train.sample(5)

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,attack_cat,label
118463,0.000009,udp,dns,INT,2,0,114,0,111111.109375,50666664.0,...,0,0,10,10,0,0,0,0,Generic,1
170467,0.000009,udp,-,INT,2,0,168,0,111111.109375,74666664.0,...,0,0,1,1,0,0,0,0,Reconnaissance,1
66310,0.000009,unas,-,INT,2,0,200,0,111111.109375,88888888.0,...,0,0,3,3,0,0,0,0,DoS,1
160418,0.000003,udp,dns,INT,2,0,114,0,333333.312500,152000000.0,...,0,0,8,8,0,0,0,0,Generic,1
77812,0.000009,unas,-,INT,2,0,200,0,111111.109375,88888888.0,...,0,0,1,1,0,0,0,0,Reconnaissance,1


In [5]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82332 entries, 0 to 82331
Data columns (total 36 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   dur                82332 non-null  float32 
 1   proto              82332 non-null  category
 2   service            82332 non-null  category
 3   state              82332 non-null  category
 4   spkts              82332 non-null  int16   
 5   dpkts              82332 non-null  int16   
 6   sbytes             82332 non-null  int32   
 7   dbytes             82332 non-null  int32   
 8   rate               82332 non-null  float32 
 9   sload              82332 non-null  float32 
 10  dload              82332 non-null  float32 
 11  sloss              82332 non-null  int16   
 12  dloss              82332 non-null  int16   
 13  sinpkt             82332 non-null  float32 
 14  dinpkt             82332 non-null  float32 
 15  sjit               82332 non-null  float32 
 16  djit

In [6]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175341 entries, 0 to 175340
Data columns (total 36 columns):
 #   Column             Non-Null Count   Dtype   
---  ------             --------------   -----   
 0   dur                175341 non-null  float32 
 1   proto              175341 non-null  category
 2   service            175341 non-null  category
 3   state              175341 non-null  category
 4   spkts              175341 non-null  int16   
 5   dpkts              175341 non-null  int16   
 6   sbytes             175341 non-null  int32   
 7   dbytes             175341 non-null  int32   
 8   rate               175341 non-null  float32 
 9   sload              175341 non-null  float32 
 10  dload              175341 non-null  float32 
 11  sloss              175341 non-null  int16   
 12  dloss              175341 non-null  int16   
 13  sinpkt             175341 non-null  float32 
 14  dinpkt             175341 non-null  float32 
 15  sjit               175341 non-null

In [7]:
print(df_test.shape)
print(df_train.shape)

(82332, 36)
(175341, 36)


In [ ]:
df_test.isnull().sum()

In [ ]:
df_train.isnull().sum()

In [ ]:
print(df_test.duplicated().sum())
print(df_train.duplicated().sum())

In [ ]:
print(df_test.columns)
print(df_train.columns)

# CODE EXAMPLE CHATGPT

In [ ]:
print(df_train['label'].value_counts())
print(df_test['label'].value_counts())

In [ ]:
print(df_train['attack_cat'].value_counts())
print(df_test['attack_cat'].value_counts())

In [ ]:
df_train["label"].value_counts().plot(kind="bar")
plt.title("Normal & Attack")
plt.xlabel("Label")
plt.ylabel("Number of samples")
plt.show()

df_train["attack_cat"].value_counts().plot(kind="bar")
plt.title("Normal & Attack")
plt.xlabel("Attack cat")
plt.ylabel("Number of samples")
plt.show()

In [ ]:
# Phân loại các cột theo kiểu dữ liệu
categorical_cols = df_train.select_dtypes(
    include=["category", "object"]
).columns.tolist()
numeric_cols = df_train.select_dtypes(
    include=["number"]
).columns.tolist()
print("Các cột categorical:")
print(categorical_cols)
print("\nCác cột numeric:")
print(numeric_cols)

In [ ]:
for col in categorical_cols:
    print({col})
    print("Số giá trị khác nhau:", df_train[col].nunique())
    print(df_train[col].value_counts().head(15))

In [ ]:
# THống kê các numric features
numeric_features = [
    col for col in numeric_cols
    if col != "label"
]

df_train[numeric_features].describe().T
# .T => dùng để tạo bảng thay vì chỉ xuất hiện 1 dòng như describe

In [ ]:
# So sánh các numerical feature giữa Normal và Attack

normal_stats = df_train[df_train["label"] == 0][numeric_features].mean()
attack_stats = df_train[df_train["label"] == 1][numeric_features].mean()

comparison = pd.DataFrame({
    "Normal_mean": normal_stats,
    "Attack_mean": attack_stats
})

comparison["Difference"] = (
    comparison["Attack_mean"] - comparison["Normal_mean"]
)

comparison.sort_values(
    by="Difference",
    key=lambda x: x.abs(),
    ascending=False
)

In [ ]:
# So sánh median và tỷ lệ giá trị bằng 0
# giữa Normal và Attack

normal_df = df_train[df_train["label"] == 0]
attack_df = df_train[df_train["label"] == 1]

comparison_median = pd.DataFrame({
    "Normal_median": normal_df[numeric_features].median(),
    "Attack_median": attack_df[numeric_features].median(),

    "Normal_zero_%": (
        (normal_df[numeric_features] == 0).mean() * 100
    ),

    "Attack_zero_%": (
        (attack_df[numeric_features] == 0).mean() * 100
    )
})

comparison_median

In [ ]:
# Phân tích categorical features theo label

for col in ["proto", "service", "state"]:
    print(f"=== {col} ===")
    
    table = pd.crosstab(
        df_train[col],
        df_train["label"],
        normalize="index"
    ) * 100
    
    table.columns = ["Normal_%", "Attack_%"]
    
    # print(table.sort_values("Normal_%", ascending=False).head(15))
    print(table.sort_values("Attack_%", ascending=False).head(15))

In [ ]:
# Kiểm tra số lượng mẫu và tỷ lệ Attack của từng categorical value

for col in ["proto", "service", "state"]:
    print(f"=== {col} ===")

    analysis = pd.crosstab(
        df_train[col],
        df_train["label"]
    )

    analysis.columns = ["Normal_count", "Attack_count"]
    analysis["Total"] = (
        analysis["Normal_count"] +
        analysis["Attack_count"]
    )

    analysis["Attack_%"] = (
        analysis["Attack_count"] /
        analysis["Total"] * 100
    )

    print(
        analysis
        .sort_values("Total", ascending=False)
        .head(15)
    )

In [ ]:
# Kiểm tra category trong train và test

for col in ["proto", "service", "state"]:

    train_values = set(df_train[col].unique())
    test_values = set(df_test[col].unique())

    only_in_train = train_values - test_values
    only_in_test = test_values - train_values

    print(f"\n{'=' * 50}")
    print(f"=== {col} ===")

    print("Số category trong train:", len(train_values))
    print("Số category trong test :", len(test_values))

    print("Chỉ xuất hiện trong train:", only_in_train)
    print("Chỉ xuất hiện trong test :", only_in_test)

In [ ]:
# Tách feature (X) và target (y)
feature_cols = [
    col for col in df_train.columns
    if col not in ["attack_cat", "label"]
]

X_train = df_train[feature_cols]
y_train = df_train["label"]

X_test = df_test[feature_cols]
y_test = df_test["label"]

print("Số lượng feature:", len(feature_cols))

print("\nX_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nX_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nDanh sách feature:")
print(feature_cols)

In [ ]:
# Chuẩn hóa các cột có dạng chữ sang dạng số
# Biến các categorical feature (proto, service, state) thành các cột 0/1
# Cấu hình encoder để bỏ qua category chưa từng thấy trong Test
categorical_features = ["proto", "service", "state"]

encoder = OneHotEncoder(
    handle_unknown="ignore", # => Xử lí các biến không có trong train sẽ được encoder tránh crash và các cột encoded tương ứng = 0
    sparse_output=False
)

X_train_cat = encoder.fit_transform( # => Encoder học category từ TRAIN
    X_train[categorical_features]
)

X_test_cat = encoder.transform( # => Test chỉ biến đổi theo những gì encoder đã học
    X_test[categorical_features]
)

print("Kích thước categorical sau encoding:")
print("X_train_cat:", X_train_cat.shape)
print("X_test_cat:", X_test_cat.shape)

In [ ]:
# Kiểm tra NaN và Inf trong numerical features
# NaN đã kiểm tra trước đó nên hiện tại chỉ cần kiểm tra inf trong 2 tập train và test
X_train_num = X_train[numeric_features]
X_test_num = X_test[numeric_features]

print("NaN trong X_train:", X_train_num.isna().sum().sum())
print("NaN trong X_test :", X_test_num.isna().sum().sum())

print("Inf trong X_train:", np.isinf(X_train_num).sum().sum())
print("Inf trong X_test :", np.isinf(X_test_num).sum().sum())

In [ ]:
# Ghép numerical + categorical features
X_train_num_array = X_train_num.to_numpy()
X_test_num_array = X_test_num.to_numpy()

X_train_processed = np.hstack([
    X_train_num_array,
    X_train_cat
])

X_test_processed = np.hstack([
    X_test_num_array,
    X_test_cat
])

print("X_train_processed:", X_train_processed.shape)
print("X_test_processed :", X_test_processed.shape)

print("\nKiểu dữ liệu:")
print(X_train_processed.dtype)

In [ ]:
# Tạo model baseline
rf_model = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

# Huấn luyện trên TRAIN
rf_model.fit(X_train_processed, y_train)

print("Đã train xong Random Forest!")

In [ ]:
# Dự đoán trên tập TEST

y_pred = rf_model.predict(X_test_processed)

print("Đã dự đoán xong!")
print("Số lượng dự đoán:", len(y_pred))

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Normal", "Attack"]
))

#         Normal  Attack
# Normal  27343    9657
# Attack  1279     44053

In [ ]:
# Kiểm tra thử feature quan trọng nhất của Random Forest => Cách mà Random Forest sử dụng feature tương đối nhiều trong dự đoán
# Tạo tên cho 186 feature sau preprocessing
encoded_feature_names = encoder.get_feature_names_out(
    categorical_features
)

all_feature_names = (
    numeric_features +
    list(encoded_feature_names)
)

feature_importance = pd.DataFrame({
    "feature": all_feature_names,
    "importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
)

print(feature_importance.head(20))

In [ ]:
top_features = feature_importance.head(15).sort_values(
    by="importance"
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_features["feature"],
    top_features["importance"]
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Top 15 Feature Importance - Random Forest")

plt.tight_layout()
plt.show()

In [ ]:
# Xác định False Positive và False Negative
fp_mask = (y_test == 0) & (y_pred == 1)
fn_mask = (y_test == 1) & (y_pred == 0)

print("False Positive:", fp_mask.sum())
print("False Negative:", fn_mask.sum())

In [ ]:
# So sánh các numerical features:
# Normal được dự đoán đúng (TN)
# với Normal bị dự đoán nhầm thành Attack (FP)
tn_mask = (y_test == 0) & (y_pred == 0)

tn_df = df_test.loc[tn_mask]
fp_df = df_test.loc[fp_mask]

comparison_fp = pd.DataFrame({
    "TN_mean": tn_df[numeric_features].mean(),
    "FP_mean": fp_df[numeric_features].mean(),

    "TN_median": tn_df[numeric_features].median(),
    "FP_median": fp_df[numeric_features].median()
})

comparison_fp

In [ ]:
# So sánh tỉ lệ attack dự đoán và thực tế
print("Thực tế:")
print(y_test.value_counts())

print("\nModel dự đoán:")
print(pd.Series(y_pred).value_counts())

In [ ]:
# Xác suất model dự đoán Attack
y_proba = rf_model.predict_proba(X_test_processed)[:, 1]

fp_probabilities = y_proba[fp_mask]

print("FP có xác suất Attack:")
print(pd.Series(fp_probabilities).describe())

In [ ]:
# So sánh categorical features giữa TN và FP
categorical_compare = ["proto", "service", "state"]

for col in categorical_compare:
    print(f"=== {col} ===")

    tn_dist = tn_df[col].value_counts(normalize=True) * 100
    fp_dist = fp_df[col].value_counts(normalize=True) * 100

    comparison = pd.DataFrame({
        "TN_%": tn_dist,
        "FP_%": fp_dist
    }).fillna(0)

    comparison["Difference_%"] = (
        comparison["FP_%"] - comparison["TN_%"]
    )

    print(
        comparison.sort_values(
            "Difference_%",
            ascending=False
        ).head(15)
    )

In [ ]:
# So sánh phân phối numerical features:
# Normal trong TRAIN với Normal trong TEST
train_normal = df_train[df_train["label"] == 0]
test_normal = df_test[df_test["label"] == 0]

distribution_compare = pd.DataFrame({
    "Train_Normal_mean": train_normal[numeric_features].mean(),
    "Test_Normal_mean": test_normal[numeric_features].mean(),

    "Train_Normal_median": train_normal[numeric_features].median(),
    "Test_Normal_median": test_normal[numeric_features].median()
})

# Độ chênh lệch mean theo %
distribution_compare["Mean_Difference_%"] = (
    (distribution_compare["Test_Normal_mean"]
     - distribution_compare["Train_Normal_mean"])
    / distribution_compare["Train_Normal_mean"].replace(0, np.nan)
    * 100
)

print("So sánh phân phối Normal: TRAIN vs TEST")
print(
    distribution_compare
    .sort_values(
        "Mean_Difference_%",
        key=lambda x: x.abs(),
        ascending=False
    )
    .head(20)
)

# => Mục đích : xem những feature mà FP có giá trị cao có phải cũng đã thay đổi phân phối giữa Train Normal và Test Normal hay không.

In [ ]:
# So sánh phân phối categorical features:
# Normal trong TRAIN với Normal trong TEST
for col in ["proto", "service", "state"]:

    print(f"=== {col} ===")

    train_dist = (
        train_normal[col]
        .value_counts(normalize=True) * 100
    )

    test_dist = (
        test_normal[col]
        .value_counts(normalize=True) * 100
    )

    comparison = pd.DataFrame({
        "Train_Normal_%": train_dist,
        "Test_Normal_%": test_dist
    }).fillna(0)

    comparison["Difference_%"] = (
        comparison["Test_Normal_%"]
        - comparison["Train_Normal_%"]
    )

    print(
        comparison
        .sort_values(
            "Difference_%",
            key=lambda x: x.abs(),
            ascending=False
        )
        .head(15)
    )

In [ ]:
# KS test:
# Kiểm tra Train Normal và Test Normal có khác phân phối hay không
ks_results = []

for col in numeric_features:

    train_values = train_normal[col].dropna()
    test_values = test_normal[col].dropna()

    statistic, p_value = ks_2samp(
        train_values,
        test_values
    )

    ks_results.append({
        "Feature": col,
        "KS_statistic": statistic,
        "p_value": p_value
    })

ks_df = pd.DataFrame(ks_results)

print("Các numerical features có khác biệt phân phối mạnh nhất:")

print(
    ks_df
    .sort_values("KS_statistic", ascending=False)
    .head(20)
)

In [ ]:
# Các feature có KS statistic cao nhất
top_features = (
    ks_df
    .sort_values("KS_statistic", ascending=False)
    .head(10)["Feature"]
    .tolist()
)

comparison = []

for col in top_features:

    train_values = train_normal[col].dropna()
    test_values = test_normal[col].dropna()

    comparison.append({
        "Feature": col,

        "Train_Mean": train_values.mean(),
        "Test_Mean": test_values.mean(),

        "Train_Median": train_values.median(),
        "Test_Median": test_values.median(),

        "Train_Q25": train_values.quantile(0.25),
        "Test_Q25": test_values.quantile(0.25),

        "Train_Q75": train_values.quantile(0.75),
        "Test_Q75": test_values.quantile(0.75),
    })

distribution_detail = pd.DataFrame(comparison)

print("Chi tiết các feature có KS statistic cao nhất:")
display(distribution_detail)

In [ ]:
# Tạo bản sao df_test để phân tích
df_test_analysis = df_test.copy()

# Đưa prediction vào DataFrame
df_test_analysis["prediction"] = y_pred

# False Positive:
# Thực tế Normal (label = 0)
# nhưng model dự đoán Attack (prediction = 1)

fp = df_test_analysis[
    (df_test_analysis["label"] == 0) &
    (df_test_analysis["prediction"] == 1)
]

# True Negative:
# Thực tế Normal
# và model dự đoán Normal

tn = df_test_analysis[
    (df_test_analysis["label"] == 0) &
    (df_test_analysis["prediction"] == 0)
]

print("Số TN:", len(tn))
print("Số FP:", len(fp))

In [ ]:
# So sánh TN và FP trên các feature có KS statistic cao
top_features = (
    ks_df
    .sort_values("KS_statistic", ascending=False)
    .head(10)["Feature"]
    .tolist()
)

comparison_fp_tn = []

for col in top_features:

    tn_values = tn[col].dropna()
    fp_values = fp[col].dropna()

    comparison_fp_tn.append({
        "Feature": col,

        "TN_Median": tn_values.median(),
        "FP_Median": fp_values.median(),

        "TN_Q25": tn_values.quantile(0.25),
        "FP_Q25": fp_values.quantile(0.25),

        "TN_Q75": tn_values.quantile(0.75),
        "FP_Q75": fp_values.quantile(0.75)
    })

fp_tn_detail = pd.DataFrame(comparison_fp_tn)

display(fp_tn_detail)

In [ ]:
# Dùng Log1p để trực quan hóa rõ ràng hơn
plot_features = ["ackdat", "tcprtt", "dload", "rate"]

for col in plot_features:

    tn_values = tn[col].dropna()
    fp_values = fp[col].dropna()

    plt.figure(figsize=(8, 4))

    plt.hist(
        np.log1p(tn_values),
        bins=50,
        alpha=0.5,
        label="TN"
    )

    plt.hist(
        np.log1p(fp_values),
        bins=50,
        alpha=0.5,
        label="FP"
    )

    plt.title(f"TN vs FP - log1p({col})")
    plt.xlabel(f"log1p({col})")
    plt.ylabel("Frequency")
    plt.legend()

    plt.show()

In [ ]:
# # Tìm các object trong notebook có khả năng là preprocessing transformer
# candidates = []

# for name, obj in globals().items():
#     if hasattr(obj, "get_feature_names_out"):
#         candidates.append(name)

# print("Các object có get_feature_names_out():")
# print(candidates)

In [ ]:
feature_names = encoder.get_feature_names_out()

print("Số feature names:", len(feature_names))
print("Số feature importance:", len(rf_model.feature_importances_))

In [ ]:
encoded_feature_names = encoder.get_feature_names_out()
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()

print("Numerical features:", len(numeric_features))
print("Encoded features:", len(encoded_feature_names))

In [ ]:
feature_names = (
    numeric_features
    + encoded_feature_names.tolist()
)

print("Total feature names:", len(feature_names))
print("RF importances:", len(rf_model.feature_importances_))

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(feature_importance.head(20))

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print("Train subset:", X_tr.shape)
print("Validation:", X_val.shape)
print("Attack rate - Train:", y_tr.mean())
print("Attack rate - Validation:", y_val.mean())

In [ ]:
categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumber of categorical features:", len(categorical_features))
print("Number of numerical features:", len(numeric_features))

In [ ]:
def run_validation_experiment(drop_features=None):

    if drop_features is None:
        drop_features = []

    # Loại feature khỏi raw data
    X_tr_exp = X_tr.drop(columns=drop_features, errors="ignore").copy()
    X_val_exp = X_val.drop(columns=drop_features, errors="ignore").copy()

    # Xác định numerical / categorical sau khi bỏ feature
    cat_cols = X_tr_exp.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    num_cols = X_tr_exp.select_dtypes(
        include=["number"]
    ).columns.tolist()

    # Fit encoder CHỈ trên training subset
    encoder_exp = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )

    X_tr_cat = encoder_exp.fit_transform(X_tr_exp[cat_cols])
    X_val_cat = encoder_exp.transform(X_val_exp[cat_cols])

    # Numerical giữ nguyên
    X_tr_num = X_tr_exp[num_cols].to_numpy()
    X_val_num = X_val_exp[num_cols].to_numpy()

    # Ghép numerical + categorical
    X_tr_processed = np.hstack([
        X_tr_num,
        X_tr_cat
    ])

    X_val_processed = np.hstack([
        X_val_num,
        X_val_cat
    ])

    # Clone chính Random Forest baseline
    rf_exp = clone(rf_model)

    rf_exp.fit(X_tr_processed, y_tr)

    y_val_pred = rf_exp.predict(X_val_processed)

    cm = confusion_matrix(y_val, y_val_pred)

    tn_val, fp_val, fn_val, tp_val = cm.ravel()

    return {
        "Accuracy": accuracy_score(y_val, y_val_pred),
        "Precision_Normal": precision_score(
            y_val, y_val_pred, pos_label=0
        ),
        "Recall_Normal": recall_score(
            y_val, y_val_pred, pos_label=0
        ),
        "Precision_Attack": precision_score(
            y_val, y_val_pred, pos_label=1
        ),
        "Recall_Attack": recall_score(
            y_val, y_val_pred, pos_label=1
        ),
        "F1_Attack": f1_score(
            y_val, y_val_pred, pos_label=1
        ),
        "TN": tn_val,
        "FP": fp_val,
        "FN": fn_val,
        "TP": tp_val
    }

In [ ]:
baseline_result = run_validation_experiment()

pd.DataFrame([baseline_result])

In [ ]:
result_no_tcp_timing = run_validation_experiment(
    drop_features=[
        "tcprtt",
        "synack",
        "ackdat"
    ]
)

pd.DataFrame([result_no_tcp_timing])

In [ ]:
result_no_tcp_seq = run_validation_experiment(
    drop_features=[
        "stcpb",
        "dtcpb"
    ]
)

pd.DataFrame([result_no_tcp_seq])

In [ ]:
result_no_tcp_all = run_validation_experiment(
    drop_features=[
        "tcprtt",
        "synack",
        "ackdat",
        "stcpb",
        "dtcpb"
    ]
)

pd.DataFrame([result_no_tcp_all])

In [ ]:
def train_and_evaluate_test(drop_features=None):

    if drop_features is None:
        drop_features = []

    # =========================
    # 1. Drop features
    # =========================
    X_train_exp = X_train.drop(
        columns=drop_features,
        errors="ignore"
    ).copy()

    X_test_exp = X_test.drop(
        columns=drop_features,
        errors="ignore"
    ).copy()

    # =========================
    # 2. Detect feature types
    # =========================
    cat_cols = X_train_exp.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    num_cols = X_train_exp.select_dtypes(
        include=["number"]
    ).columns.tolist()

    # =========================
    # 3. Fit encoder ONLY on full TRAIN
    # =========================
    encoder_final = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )

    X_train_cat = encoder_final.fit_transform(
        X_train_exp[cat_cols]
    )

    X_test_cat = encoder_final.transform(
        X_test_exp[cat_cols]
    )

    # =========================
    # 4. Numerical features
    # =========================
    X_train_num = X_train_exp[num_cols].to_numpy()
    X_test_num = X_test_exp[num_cols].to_numpy()

    # =========================
    # 5. Combine
    # =========================
    X_train_processed = np.hstack([
        X_train_num,
        X_train_cat
    ])

    X_test_processed = np.hstack([
        X_test_num,
        X_test_cat
    ])

    print("Train processed:", X_train_processed.shape)
    print("Test processed:", X_test_processed.shape)

    # =========================
    # 6. Train Random Forest
    # =========================
    model_final = clone(rf_model)

    model_final.fit(
        X_train_processed,
        y_train
    )

    # =========================
    # 7. Predict TEST
    # =========================
    y_test_pred = model_final.predict(
        X_test_processed
    )

    # =========================
    # 8. Metrics
    # =========================
    cm = confusion_matrix(
        y_test,
        y_test_pred
    )

    tn, fp, fn, tp = cm.ravel()

    result = {
        "Accuracy": accuracy_score(
            y_test,
            y_test_pred
        ),

        "Precision_Normal": precision_score(
            y_test,
            y_test_pred,
            pos_label=0
        ),

        "Recall_Normal": recall_score(
            y_test,
            y_test_pred,
            pos_label=0
        ),

        "Precision_Attack": precision_score(
            y_test,
            y_test_pred,
            pos_label=1
        ),

        "Recall_Attack": recall_score(
            y_test,
            y_test_pred,
            pos_label=1
        ),

        "F1_Attack": f1_score(
            y_test,
            y_test_pred,
            pos_label=1
        ),

        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

    return result, model_final, encoder_final

In [ ]:
# Đánh giá Baseline trên test
baseline_test, baseline_model_test, baseline_encoder_test = (
    train_and_evaluate_test()
)

pd.DataFrame([baseline_test])

In [ ]:
# Đánh giá Cantidate trên Test
drop_features_final = [
    "tcprtt",
    "synack",
    "ackdat",
    "stcpb",
    "dtcpb"
]

candidate_test, candidate_model_test, candidate_encoder_test = (
    train_and_evaluate_test(
        drop_features=drop_features_final
    )
)

pd.DataFrame([candidate_test])

In [ ]:
#  Test thử 2 model khi đứng cạnh nhau
comparison_test = pd.DataFrame([
    {
        "Model": "Baseline",
        **baseline_test
    },
    {
        "Model": "Candidate - remove 5 features",
        **candidate_test
    }
])

display(comparison_test)

In [ ]:
# Kiểm tra lại các Columns sau khi đã loại bỏ các candidate => ("tcprtt","synack","ackdat","stcpb","dtcpb")
X_train.drop(columns=drop_features_final).columns

# Kiểm tra lại model, Multiclass và tiến hành huấn luyện model

In [ ]:
# # Tìm các DataFrame hiện đang có cột attack_cat

# for name, obj in globals().items():
#     if isinstance(obj, pd.DataFrame):
#         if "attack_cat" in obj.columns:
#             print(
#                 f"{name}: "
#                 f"shape={obj.shape}, "
#                 f"attack_cat found"
#             )

In [ ]:
# Phân bố các lớp attack_cat trong TRAIN

attack_cat_counts = df_train["attack_cat"].value_counts(dropna=False)

print("=== Phân bố attack_cat trong TRAIN ===")
print(attack_cat_counts)

print("\n=== Tỷ lệ (%) ===")
print(
    (df_train["attack_cat"].value_counts(normalize=True, dropna=False) * 100)
    .round(3)
)

In [ ]:
# Độ khớp giữa X_train và df_train
print("X_train shape:", X_train.shape)
print("df_train shape:", df_train.shape)

print("\nIndex X_train == df_train:",
      X_train.index.equals(df_train.index))

print("\nCác feature trong X_train có nằm trong df_train không:",
      set(X_train.columns).issubset(df_train.columns))

In [ ]:
y_multiclass = df_train.loc[X_train.index, "attack_cat"]

In [ ]:
# ==========================================
# TẠO DATASET MULTICLASS CHỈ CHO ATTACK
# ==========================================

# Lấy attack_cat tương ứng với X_train
y_attack_cat = df_train.loc[X_train.index, "attack_cat"]

# Chỉ giữ các dòng thực sự là Attack
attack_mask = y_attack_cat != "Normal"

X_attack = X_train.loc[attack_mask].copy()
y_attack = y_attack_cat.loc[attack_mask].copy()

print("=== Multiclass Attack Dataset ===")
print("X_attack shape:", X_attack.shape)
print("y_attack shape:", y_attack.shape)

print("\n=== Phân bố các Attack class ===")
print(y_attack.value_counts())

print("\n=== Tỷ lệ (%) ===")
print(
    (y_attack.value_counts(normalize=True) * 100)
    .round(3)
)

In [ ]:
# ==========================================
# MULTICLASS TRAIN / VALIDATION SPLIT
# ==========================================

X_attack_train, X_attack_val, y_attack_train, y_attack_val = train_test_split(
    X_attack,
    y_attack,
    test_size=0.20,
    random_state=42,
    stratify=y_attack
)

print("=== Multiclass split ===")
print("Train:", X_attack_train.shape)
print("Validation:", X_attack_val.shape)

print("\n=== Train class distribution ===")
print(y_attack_train.value_counts())

print("\n=== Validation class distribution ===")
print(y_attack_val.value_counts())

# Train Multiclass model

In [ ]:
# ==========================================
# MULTICLASS FEATURE SET
# ==========================================

drop_features_multiclass = [
    "tcprtt",
    "synack",
    "ackdat",
    "stcpb",
    "dtcpb"
]

X_attack_train_mc = X_attack_train.drop(
    columns=drop_features_multiclass
).copy()

X_attack_val_mc = X_attack_val.drop(
    columns=drop_features_multiclass
).copy()

print("Original features:", X_attack_train.shape[1])
print("Features after drop:", X_attack_train_mc.shape[1])

print("\nCategorical features:")
print(
    X_attack_train_mc.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()
)

print("\nNumerical features:",
      X_attack_train_mc.select_dtypes(
          exclude=["object", "category"]
      ).shape[1])

In [ ]:
# ==========================================
# MULTICLASS PREPROCESSOR
# ==========================================

categorical_features_mc = [
    "proto",
    "service",
    "state"
]

numerical_features_mc = [
    col for col in X_attack_train_mc.columns
    if col not in categorical_features_mc
]

preprocessor_mc = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features_mc
        ),
        (
            "num",
            "passthrough",
            numerical_features_mc
        )
    ]
)

# Fit CHỈ trên multiclass TRAIN
X_attack_train_encoded = preprocessor_mc.fit_transform(
    X_attack_train_mc
)

# Validation chỉ transform, KHÔNG fit lại
X_attack_val_encoded = preprocessor_mc.transform(
    X_attack_val_mc
)

print("=== Multiclass preprocessing ===")
print("Train encoded shape:", X_attack_train_encoded.shape)
print("Validation encoded shape:", X_attack_val_encoded.shape)

In [ ]:
# ==========================================
# MULTICLASS RANDOM FOREST - BASELINE
# ==========================================

rf_multiclass = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

print("Training multiclass Random Forest...")

rf_multiclass.fit(
    X_attack_train_encoded,
    y_attack_train
)

print("Training completed!")

In [ ]:
# ==========================================
# MULTICLASS VALIDATION EVALUATION
# ==========================================

y_attack_val_pred = rf_multiclass.predict(
    X_attack_val_encoded
)

accuracy_mc = accuracy_score(
    y_attack_val,
    y_attack_val_pred
)

macro_f1_mc = f1_score(
    y_attack_val,
    y_attack_val_pred,
    average="macro"
)

weighted_f1_mc = f1_score(
    y_attack_val,
    y_attack_val_pred,
    average="weighted"
)

print("=== MULTICLASS VALIDATION ===")
print(f"Accuracy   : {accuracy_mc:.6f}")
print(f"Macro-F1   : {macro_f1_mc:.6f}")
print(f"Weighted-F1: {weighted_f1_mc:.6f}")

print("\n=== Classification Report ===")

print(
    classification_report(
        y_attack_val,
        y_attack_val_pred,
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ==========================================
# MULTICLASS CONFUSION MATRIX
# ==========================================

class_names = rf_multiclass.classes_

cm = confusion_matrix(
    y_attack_val,
    y_attack_val_pred,
    labels=class_names
)

print("Class order:")
print(class_names)

print("\nConfusion Matrix:")
print(cm)

plt.figure(figsize=(10, 8))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(
    xticks_rotation=45,
    values_format="d"
)

plt.title("Multiclass Attack - Validation Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# RANDOM FOREST - BALANCED SUBSAMPLE
# ==========================================

rf_multiclass_bs = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

print("Training Random Forest (balanced_subsample)...")

rf_multiclass_bs.fit(
    X_attack_train_encoded,
    y_attack_train
)

print("Training completed!")

# Validation prediction
y_attack_val_pred_bs = rf_multiclass_bs.predict(
    X_attack_val_encoded
)

# Metrics
accuracy_bs = accuracy_score(
    y_attack_val,
    y_attack_val_pred_bs
)

macro_f1_bs = f1_score(
    y_attack_val,
    y_attack_val_pred_bs,
    average="macro"
)

weighted_f1_bs = f1_score(
    y_attack_val,
    y_attack_val_pred_bs,
    average="weighted"
)

print("\n=== RANDOM FOREST BALANCED SUBSAMPLE ===")
print(f"Accuracy    : {accuracy_bs:.6f}")
print(f"Macro-F1    : {macro_f1_bs:.6f}")
print(f"Weighted-F1 : {weighted_f1_bs:.6f}")

print("\n=== Classification Report ===")

print(
    classification_report(
        y_attack_val,
        y_attack_val_pred_bs,
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ==========================================
# FINAL MULTICLASS MODEL
# Train trên TOÀN BỘ Attack training data
# ==========================================
# ------------------------------------------
# 1. Feature set
# ------------------------------------------

drop_features_final_mc = [
    "tcprtt",
    "synack",
    "ackdat",
    "stcpb",
    "dtcpb"
]

X_attack_full_mc = X_attack.drop(
    columns=drop_features_final_mc
).copy()

# ------------------------------------------
# 2. Features
# ------------------------------------------

categorical_features_final_mc = [
    "proto",
    "service",
    "state"
]

numerical_features_final_mc = [
    col for col in X_attack_full_mc.columns
    if col not in categorical_features_final_mc
]

# ------------------------------------------
# 3. Final preprocessing
# ------------------------------------------

final_preprocessor_mc = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features_final_mc
        ),
        (
            "num",
            "passthrough",
            numerical_features_final_mc
        )
    ]
)

X_attack_full_encoded = final_preprocessor_mc.fit_transform(
    X_attack_full_mc
)

print("=== FINAL MULTICLASS PREPROCESSING ===")
print("Raw features:", X_attack_full_mc.shape)
print("Encoded features:", X_attack_full_encoded.shape)

# ------------------------------------------
# 4. Final Random Forest
# ------------------------------------------

final_multiclass_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

print("\nTraining FINAL multiclass model...")

final_multiclass_model.fit(
    X_attack_full_encoded,
    y_attack
)

print("Final multiclass training completed!")

In [ ]:
# ==========================================
# FINAL MULTICLASS TEST EVALUATION
# ==========================================

# 1. Lấy Attack samples từ TEST
test_attack_mask = df_test["label"] == 1

X_test_attack = df_test.loc[
    test_attack_mask,
    X_attack.columns
].copy()

y_test_attack = df_test.loc[
    test_attack_mask,
    "attack_cat"
].copy()

print("=== FINAL MULTICLASS TEST DATA ===")
print("X_test_attack:", X_test_attack.shape)
print("y_test_attack:", y_test_attack.shape)

print("\nAttack class distribution:")
print(y_test_attack.value_counts())

In [ ]:
# ==========================================
# FINAL MULTICLASS TEST EVALUATION
# ==========================================
# 1. Drop đúng 5 features
X_test_attack_mc = X_test_attack.drop(
    columns=drop_features_final_mc
).copy()

# 2. Transform bằng FINAL encoder
#    KHÔNG fit lại trên Test
X_test_attack_encoded = final_preprocessor_mc.transform(
    X_test_attack_mc
)

print("=== TEST PREPROCESSING ===")
print(
    "Test encoded shape:",
    X_test_attack_encoded.shape
)

# 3. Prediction
y_test_attack_pred = final_multiclass_model.predict(
    X_test_attack_encoded
)

# 4. Metrics
accuracy_test_mc = accuracy_score(
    y_test_attack,
    y_test_attack_pred
)

macro_f1_test_mc = f1_score(
    y_test_attack,
    y_test_attack_pred,
    average="macro"
)

weighted_f1_test_mc = f1_score(
    y_test_attack,
    y_test_attack_pred,
    average="weighted"
)

print("\n=== FINAL MULTICLASS TEST ===")
print(f"Accuracy    : {accuracy_test_mc:.6f}")
print(f"Macro-F1    : {macro_f1_test_mc:.6f}")
print(f"Weighted-F1 : {weighted_f1_test_mc:.6f}")

print("\n=== Classification Report ===")

print(
    classification_report(
        y_test_attack,
        y_test_attack_pred,
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ==========================================
# FINAL MULTICLASS CONFUSION MATRIX
# ==========================================
labels_mc = [
    "Analysis",
    "Backdoor",
    "DoS",
    "Exploits",
    "Fuzzers",
    "Generic",
    "Reconnaissance",
    "Shellcode",
    "Worms"
]

fig, ax = plt.subplots(figsize=(10, 8))

ConfusionMatrixDisplay.from_predictions(
    y_test_attack,
    y_test_attack_pred,
    labels=labels_mc,
    display_labels=labels_mc,
    xticks_rotation=45,
    cmap="Blues",
    ax=ax
)

ax.set_title("Final Multiclass Test - Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# CHECK FINAL OBJECTS
# ==========================================
required_objects = [
    "candidate_model_test",
    "candidate_encoder_test",
    "final_multiclass_model",
    "final_preprocessor_mc",
    "drop_features_final_mc"
]

print("=== FINAL OBJECT CHECK ===")

for name in required_objects:
    if name in globals():
        obj = globals()[name]
        print(f"✓ {name:<30} {type(obj).__name__}")
    else:
        print(f"✗ {name:<30} NOT FOUND")

In [ ]:
# ==========================================
# IDS METADATA
# ==========================================
# Tránh cho featrures bị loss
# Binary model sử dụng 29 features
binary_drop_features = [
    "tcprtt",
    "synack",
    "ackdat",
    "stcpb",
    "dtcpb"
]

# Multiclass cũng sử dụng cùng feature set
multiclass_drop_features = [
    "tcprtt",
    "synack",
    "ackdat",
    "stcpb",
    "dtcpb"
]

# Danh sách feature gốc
original_features = list(X_train.columns)

# Feature thực sự đưa vào binary model
binary_features = [
    col for col in original_features
    if col not in binary_drop_features
]

# Feature thực sự đưa vào multiclass model
multiclass_features = [
    col for col in original_features
    if col not in multiclass_drop_features
]

# Class của multiclass model
multiclass_classes = list(
    final_multiclass_model.classes_
)

ids_metadata = {
    "original_features": original_features,

    "binary_features": binary_features,
    "binary_drop_features": binary_drop_features,

    "multiclass_features": multiclass_features,
    "multiclass_drop_features": multiclass_drop_features,

    "multiclass_classes": multiclass_classes,

    "binary_model_type": type(candidate_model_test).__name__,
    "multiclass_model_type": type(final_multiclass_model).__name__
}

print("=== IDS METADATA ===")
print("Original features :", len(original_features))
print("Binary features   :", len(binary_features))
print("Multiclass features:", len(multiclass_features))

print("\nMulticlass classes:")
print(multiclass_classes)

In [ ]:
# ==========================================
# SAVE FINAL IDS ARTIFACTS
# ==========================================
MODEL_DIR = "ids_model"

os.makedirs(MODEL_DIR, exist_ok=True)

# Binary
joblib.dump(
    candidate_model_test,
    os.path.join(MODEL_DIR, "binary_model.joblib")
)

joblib.dump(
    candidate_encoder_test,
    os.path.join(MODEL_DIR, "binary_encoder.joblib")
)

# Multiclass
joblib.dump(
    final_multiclass_model,
    os.path.join(MODEL_DIR, "multiclass_model.joblib")
)

joblib.dump(
    final_preprocessor_mc,
    os.path.join(MODEL_DIR, "multiclass_encoder.joblib")
)

# Metadata
joblib.dump(
    ids_metadata,
    os.path.join(MODEL_DIR, "metadata.joblib")
)

print("=== IDS ARTIFACTS SAVED ===")

for filename in os.listdir(MODEL_DIR):
    path = os.path.join(MODEL_DIR, filename)
    print(f"✓ {filename:<30} {os.path.getsize(path) / 1024:.1f} KB")

In [ ]:
# ==========================================
# IDS PREDICTION ENGINE
# ==========================================
def predict_ids(input_df):
    """
    Two-stage IDS prediction.

    Stage 1:
        Normal vs Attack

    Stage 2:
        If Attack -> classify attack category
    """

    # --------------------------------------
    # Validate input
    # --------------------------------------

    if not isinstance(input_df, pd.DataFrame):
        raise TypeError(
            "input_df must be a pandas DataFrame"
        )

    missing_features = [
        feature
        for feature in original_features
        if feature not in input_df.columns
    ]

    if missing_features:
        raise ValueError(
            f"Missing features: {missing_features}"
        )

    # --------------------------------------
    # Preserve only original features
    # --------------------------------------

    X_input = input_df[
        original_features
    ].copy()

    # --------------------------------------
    # Remove unused features
    # --------------------------------------

    X_binary = X_input.drop(
        columns=binary_drop_features
    )
    
    # --------------------------------------
    # STAGE 1 - Binary preprocessing
    # --------------------------------------
    
    # Lấy danh sách categorical features
    binary_cat_features = (
        candidate_encoder_test.feature_names_in_.tolist()
    )
    
    # Các feature số, giữ nguyên thứ tự cột đầu vào
    binary_num_features = [
        col for col in X_binary.columns
        if col not in binary_cat_features
    ]
    
    # Xác nhận đầu vào có đủ feature
    assert set(binary_cat_features).issubset(X_binary.columns)
    
    # Xử lý numerical features
    X_binary_num = X_binary[
        binary_num_features
    ].to_numpy()
    
    # Encode categorical features
    X_binary_cat = candidate_encoder_test.transform(
        X_binary[binary_cat_features]
    )
    
    # Ghép theo đúng thứ tự khi train:
    # Numerical trước, One-Hot encoded categorical sau
    X_binary_encoded = np.hstack([
        X_binary_num,
        X_binary_cat
    ])
    
    # Kiểm tra số feature trước khi predict
    expected_features = candidate_model_test.n_features_in_
    
    print("Binary input shape:", X_binary.shape)
    print("Encoded input shape:", X_binary_encoded.shape)
    print("Model expects:", expected_features)
    
    assert X_binary_encoded.shape[1] == expected_features, (
        f"Feature mismatch: encoded={X_binary_encoded.shape[1]}, "
        f"expected={expected_features}"
    )
    
    # Binary prediction
    binary_pred = candidate_model_test.predict(
        X_binary_encoded
    )

    # --------------------------------------
    # Binary prediction probabilities
    # --------------------------------------
    
    if hasattr(candidate_model_test, "predict_proba"):
        binary_prob = candidate_model_test.predict_proba(
            X_binary_encoded
        )
    else:
        binary_prob = None
    
    # --------------------------------------
    # Initialize result containers
    # --------------------------------------
    
    results = []
    attack_indices = []


    # --------------------------------------
    # Process each sample
    # --------------------------------------
    for i, pred in enumerate(binary_pred):

        if pred == 0:

            result = {
                "prediction": "Normal",
                "attack_type": None,
                "binary_prediction": int(pred),
                "binary_confidence": (
                    float(np.max(binary_prob[i]))
                    if binary_prob is not None
                    else None
                ),
                "attack_confidence": None
            }

            results.append(result)

        else:
            attack_indices.append(i)

            # Temporary result
            results.append(None)

    # --------------------------------------
    # STAGE 2 - Multiclass
    # --------------------------------------

    if attack_indices:

        X_attack = X_input.iloc[
            attack_indices
        ].drop(
            columns=multiclass_drop_features
        )

        X_attack_encoded = final_preprocessor_mc.transform(
            X_attack
        )

        multiclass_pred = final_multiclass_model.predict(
            X_attack_encoded
        )

        if hasattr(
            final_multiclass_model,
            "predict_proba"
        ):
            multiclass_prob = (
                final_multiclass_model.predict_proba(
                    X_attack_encoded
                )
            )
        else:
            multiclass_prob = None

        for j, original_index in enumerate(
            attack_indices
        ):

            attack_type = multiclass_pred[j]

            result = {
                "prediction": "Attack",
                "attack_type": attack_type,
                "binary_prediction": 1,
                "binary_confidence": (
                    float(np.max(binary_prob[original_index]))
                    if binary_prob is not None
                    else None
                ),
                "attack_confidence": (
                    float(np.max(multiclass_prob[j]))
                    if multiclass_prob is not None
                    else None
                )
            }

            results[original_index] = result

    return pd.DataFrame(results)

In [ ]:

# Lấy 20 mẫu dữ liệu để thử IDS
sample_input = df_test[original_features].head(20).copy()

# Chạy hệ thống dự đoán
sample_results = predict_ids(sample_input)

# Hiển thị kết quả
display(sample_results)


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)
import numpy as np
import pandas as pd

# ==========================================
# STEP 2 - END-TO-END IDS EVALUATION
# ==========================================

# 1. Validate test data
required_cols = (
    list(original_features) + ["label", "attack_cat"]
)
missing = [
    col for col in required_cols
    if col not in df_test.columns
]

if missing:
    raise ValueError(f"Missing test columns: {missing}")

# IMPORTANT:
# df_test must be unseen by BOTH models during training
# and must not have been used to select model settings.

eval_df = df_test.copy()

# Ground-truth binary labels: 0=Normal, 1=Attack
y_true_binary = eval_df["label"].astype(int)

# 2. Run the complete two-stage IDS
pred_df = predict_ids(
    eval_df[original_features].copy()
)

y_pred_binary = pred_df["binary_prediction"].astype(int)

# ==========================================
# A. STAGE 1 - BINARY EVALUATION
# ==========================================

print("=" * 55)
print("STAGE 1 - BINARY CLASSIFICATION")
print("=" * 55)

print("Accuracy:",
      round(accuracy_score(y_true_binary, y_pred_binary), 4))

print("\nClassification report:")
print(classification_report(
    y_true_binary,
    y_pred_binary,
    labels=[0, 1],
    target_names=["Normal", "Attack"],
    digits=4,
    zero_division=0
))

print("Confusion matrix [Normal, Attack]:")
print(confusion_matrix(
    y_true_binary,
    y_pred_binary,
    labels=[0, 1]
))

# ==========================================
# B. END-TO-END MULTICLASS EVALUATION
# ==========================================

# Ground truth:
# Normal traffic -> "Normal"
# Attack traffic -> its attack_cat label
y_true_final = np.where(
    y_true_binary.to_numpy() == 0,
    "Normal",
    eval_df["attack_cat"].astype(str).to_numpy()
)

# Final pipeline prediction
y_pred_final = np.where(
    pred_df["binary_prediction"].to_numpy() == 0,
    "Normal",
    pred_df["attack_type"].fillna("Unknown").astype(str).to_numpy()
)

all_labels = sorted(
    set(y_true_final) | set(y_pred_final)
)

print("\n" + "=" * 55)
print("END-TO-END IDS - FINAL CLASSIFICATION")
print("=" * 55)

print("Accuracy:",
      round(accuracy_score(y_true_final, y_pred_final), 4))

print("Macro-F1:",
      round(f1_score(
          y_true_final,
          y_pred_final,
          labels=all_labels,
          average="macro",
          zero_division=0
      ), 4))

print("Weighted-F1:",
      round(f1_score(
          y_true_final,
          y_pred_final,
          labels=all_labels,
          average="weighted",
          zero_division=0
      ), 4))

print("\nClassification report:")
print(classification_report(
    y_true_final,
    y_pred_final,
    labels=all_labels,
    digits=4,
    zero_division=0
))

print("Confusion matrix labels:")
print(all_labels)

print("\nEnd-to-end confusion matrix:")
print(confusion_matrix(
    y_true_final,
    y_pred_final,
    labels=all_labels
))

# ==========================================
# C. SAVE RESULTS FOR LATER ANALYSIS
# ==========================================

ids_eval_results = eval_df[
    ["label", "attack_cat"]
].copy()

ids_eval_results["true_final"] = y_true_final
ids_eval_results["predicted_final"] = y_pred_final
ids_eval_results["binary_confidence"] = (
    pred_df["binary_confidence"].to_numpy()
)
ids_eval_results["attack_confidence"] = (
    pred_df["attack_confidence"].to_numpy()
)

print("\nEvaluation completed.")
print("Evaluated samples:", len(ids_eval_results))

display(ids_eval_results.head(10))


In [ ]:

# Tìm các DataFrame có liên quan đến kết quả IDS
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = obj.columns.tolist()

        if any(
            key in cols
            for key in [
                "true_final",
                "predicted_final",
                "true_label",
                "predicted_label",
                "attack_cat"
            ]
        ):
            print(f"\nDataFrame: {name}")
            print("Shape:", obj.shape)
            print("Columns:", cols)


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import pandas as pd

# Dùng đúng DataFrame kết quả IDS
results = ids_eval_results.copy()

print("=== DATA VALIDATION ===")
print("Shape:", results.shape)
print("Columns:", results.columns.tolist())
print("Missing true labels:", results["true_final"].isna().sum())
print("Missing predictions:", results["predicted_final"].isna().sum())

# Loại bỏ mẫu thiếu nhãn hoặc dự đoán, nếu có
results = results.dropna(
    subset=["true_final", "predicted_final"]
)

y_true = results["true_final"].astype(str)
y_pred = results["predicted_final"].astype(str)

print("\nEvaluated samples:", len(results))

print("\n=== VERIFIED END-TO-END METRICS ===")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(
    "Macro-F1:",
    f"{f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}"
)
print(
    "Weighted-F1:",
    f"{f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}"
)

labels = sorted(set(y_true) | set(y_pred))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(
    y_true,
    y_pred,
    labels=labels,
    zero_division=0,
    digits=4
))

print("\n=== CONFUSION MATRIX ===")
cm = confusion_matrix(y_true, y_pred, labels=labels)

display(pd.DataFrame(
    cm,
    index=[f"True: {x}" for x in labels],
    columns=[f"Pred: {x}" for x in labels]
))


In [ ]:

# ==========================================
# IDS ERROR ANALYSIS - END TO END
# ==========================================

import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.metrics import classification_report

# 1. Verify evaluation data
error_df = ids_eval_results.copy()

required_cols = ["true_final", "predicted_final"]
missing = [c for c in required_cols if c not in error_df.columns]

if missing:
    raise ValueError(
        f"Missing columns: {missing}. "
        f"Available columns: {error_df.columns.tolist()}"
    )

error_df = error_df.dropna(
    subset=required_cols
).copy()

error_df["true_final"] = error_df["true_final"].astype(str)
error_df["predicted_final"] = error_df["predicted_final"].astype(str)

error_df["is_correct"] = (
    error_df["true_final"] == error_df["predicted_final"]
)

print("=== ERROR ANALYSIS OVERVIEW ===")
print("Evaluated samples:", len(error_df))
print("Correct predictions:", int(error_df["is_correct"].sum()))
print("Incorrect predictions:", int((~error_df["is_correct"]).sum()))
print("Error rate:", round((~error_df["is_correct"]).mean(), 4))

# 2. Per-class performance
print("\n=== PER-CLASS PERFORMANCE ===")

print(classification_report(
    error_df["true_final"],
    error_df["predicted_final"],
    labels=sorted(
        set(error_df["true_final"]) |
        set(error_df["predicted_final"])
    ),
    zero_division=0,
    digits=4
))

# 3. Find classes that are often misclassified
print("\n=== ERRORS BY TRUE CLASS ===")

errors_only = error_df.loc[
    ~error_df["is_correct"]
].copy()

error_by_class = (
    error_df.groupby("true_final")
    .agg(
        total_samples=("is_correct", "size"),
        correct=("is_correct", "sum")
    )
)

error_by_class["incorrect"] = (
    error_by_class["total_samples"]
    - error_by_class["correct"]
)

error_by_class["error_rate"] = (
    error_by_class["incorrect"]
    / error_by_class["total_samples"]
)

display(
    error_by_class.sort_values(
        "error_rate", ascending=False
    ).round(4)
)

# 4. Which classes get confused with which?
print("\n=== TOP MISCLASSIFICATION PAIRS ===")

if len(errors_only) > 0:
    error_pairs = (
        errors_only.groupby(
            ["true_final", "predicted_final"]
        )
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    display(error_pairs.head(20))
else:
    print("No misclassified samples found.")

# 5. Inspect individual wrong predictions
print("\n=== EXAMPLE MISCLASSIFIED SAMPLES ===")

if len(errors_only) > 0:
    cols_to_show = [
        c for c in [
            "label",
            "attack_cat",
            "true_final",
            "predicted_final",
            "binary_confidence",
            "attack_confidence"
        ]
        if c in errors_only.columns
    ]

    display(errors_only[cols_to_show].head(20))


In [ ]:

# ==========================================
# IDS ERROR ANALYSIS - STAGE 1 vs STAGE 2
# ==========================================

import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

df_err = ids_eval_results.copy()

required = [
    "label",
    "attack_cat",
    "predicted_final"
]

missing = [c for c in required if c not in df_err.columns]

if missing:
    raise ValueError(
        f"Missing columns: {missing}\n"
        f"Available columns: {df_err.columns.tolist()}"
    )

df_err = df_err.dropna(subset=required).copy()

# ==========================================
# STAGE 1 - BINARY ERROR ANALYSIS
# ==========================================

# Ground truth: 0 = Normal, 1 = Attack
df_err["true_binary"] = pd.to_numeric(
    df_err["label"], errors="raise"
).astype(int)

if not set(df_err["true_binary"].unique()).issubset({0, 1}):
    raise ValueError("label must contain only 0 and 1.")

# Infer binary decision from final output:
# predicted_final == Normal -> Binary predicted Normal
# otherwise -> Binary predicted Attack
df_err["pred_binary"] = (
    df_err["predicted_final"].astype(str) != "Normal"
).astype(int)

print("=" * 55)
print("STAGE 1 - BINARY ERROR ANALYSIS")
print("=" * 55)

print("\nBinary confusion matrix")
print("Rows = true [Normal, Attack]")
print("Columns = predicted [Normal, Attack]")

cm_binary = confusion_matrix(
    df_err["true_binary"],
    df_err["pred_binary"],
    labels=[0, 1]
)

display(pd.DataFrame(
    cm_binary,
    index=["True Normal", "True Attack"],
    columns=["Pred Normal", "Pred Attack"]
))

print("\nBinary classification report")
print(classification_report(
    df_err["true_binary"],
    df_err["pred_binary"],
    labels=[0, 1],
    target_names=["Normal", "Attack"],
    zero_division=0,
    digits=4
))

# False positives: Normal incorrectly flagged as Attack
false_positive = df_err[
    (df_err["true_binary"] == 0) &
    (df_err["pred_binary"] == 1)
]

# False negatives: Attack incorrectly classified as Normal
false_negative = df_err[
    (df_err["true_binary"] == 1) &
    (df_err["pred_binary"] == 0)
]

print("False positives (Normal -> Attack):", len(false_positive))
print("False negatives (Attack -> Normal):", len(false_negative))

print("\nTrue attack classes missed by Binary")
display(
    false_negative["attack_cat"]
    .value_counts()
    .rename_axis("true_attack_class")
    .reset_index(name="missed_samples")
)

# ==========================================
# STAGE 2 - MULTICLASS ERROR ANALYSIS
# Conditional on true attacks detected as Attack
# ==========================================

stage2_df = df_err[
    (df_err["true_binary"] == 1) &
    (df_err["pred_binary"] == 1)
].copy()

# Exclude Normal from the true attack categories
stage2_df = stage2_df[
    stage2_df["attack_cat"].astype(str) != "Normal"
]

print("\n" + "=" * 55)
print("STAGE 2 - MULTICLASS CONDITIONAL ANALYSIS")
print("=" * 55)

print("Attacks reaching Stage 2:", len(stage2_df))

if len(stage2_df) > 0:
    y_true_mc = stage2_df["attack_cat"].astype(str)
    y_pred_mc = stage2_df["predicted_final"].astype(str)

    labels_mc = sorted(
        set(y_true_mc) | set(y_pred_mc)
    )

    print(
        "\nConditional accuracy:",
        round(accuracy_score(y_true_mc, y_pred_mc), 4)
    )

    print("\nConditional classification report")
    print(classification_report(
        y_true_mc,
        y_pred_mc,
        labels=labels_mc,
        zero_division=0,
        digits=4
    ))

    wrong_mc = stage2_df[
        y_true_mc != y_pred_mc
    ].copy()

    print("\nTop Stage 2 misclassification pairs")

    if len(wrong_mc) > 0:
        pairs = (
            wrong_mc.groupby(
                ["attack_cat", "predicted_final"]
            )
            .size()
            .reset_index(name="count")
            .sort_values("count", ascending=False)
        )

        display(pairs.head(20))
    else:
        print("No conditional multiclass errors.")
else:
    print("No eligible attack samples for Stage 2 analysis.")


In [ ]:
# ==========================================
# FINAL PIPELINE COMPARISON
# 29 FEATURES vs 14 FEATURES
# ==========================================
print("=" * 70)
print("FINAL PIPELINE COMPARISON")
print("=" * 70)

# --------------------------------------------------
# 1. KẾT QUẢ PIPELINE 29 FEATURES
#    Lấy từ kết quả evaluation/error analysis vừa chạy
# --------------------------------------------------

pipeline_29 = {
    "Stage 1 Accuracy": 0.8714,
    "Stage 1 Recall Normal": 0.7504,
    "Stage 1 Recall Attack": 0.9702,
    "Stage 1 Macro-F1": 0.8662,

    "Stage 2 Accuracy": 0.7596,
    "Stage 2 Macro-F1": 0.5107,
    "Stage 2 Weighted-F1": 0.7912,

    "End-to-End Accuracy": 0.7430,
    "End-to-End Macro-F1": 0.5006,
    "End-to-End Weighted-F1": 0.7785
}


# --------------------------------------------------
# 2. KẾT QUẢ PIPELINE 14 FEATURES
#    Lấy từ cell Claude đã train
# --------------------------------------------------

pipeline_14 = {
    "Stage 1 Accuracy": 0.908,
    "Stage 1 Recall Normal": 0.863,
    "Stage 1 Recall Attack": 0.945,
    "Stage 1 Macro-F1": 0.907,

    "Stage 2 Accuracy": 0.762,
    "Stage 2 Macro-F1": 0.564,
    "Stage 2 Weighted-F1": 0.796,

    # KHÔNG sử dụng 0.404 để so sánh trực tiếp
    # vì cần xác minh lại cách Claude tính end-to-end.
    "End-to-End Accuracy": np.nan,
    "End-to-End Macro-F1": np.nan,
    "End-to-End Weighted-F1": np.nan
}


# --------------------------------------------------
# 3. TẠO BẢNG SO SÁNH
# --------------------------------------------------

comparison_df = pd.DataFrame({
    "Metric": list(pipeline_29.keys()),
    "29-feature": list(pipeline_29.values()),
    "14-feature": list(pipeline_14.values())
})

comparison_df["Difference (14 - 29)"] = (
    comparison_df["14-feature"]
    - comparison_df["29-feature"]
)

display(comparison_df)


# --------------------------------------------------
# 4. STAGE 1 COMPARISON
# --------------------------------------------------

print("\n" + "=" * 70)
print("STAGE 1 - BINARY CLASSIFICATION")
print("=" * 70)

stage1_metrics = [
    "Stage 1 Accuracy",
    "Stage 1 Recall Normal",
    "Stage 1 Recall Attack",
    "Stage 1 Macro-F1"
]

display(
    comparison_df[
        comparison_df["Metric"].isin(stage1_metrics)
    ]
)


# --------------------------------------------------
# 5. STAGE 2 COMPARISON
# --------------------------------------------------

print("\n" + "=" * 70)
print("STAGE 2 - MULTICLASS CLASSIFICATION")
print("=" * 70)

stage2_metrics = [
    "Stage 2 Accuracy",
    "Stage 2 Macro-F1",
    "Stage 2 Weighted-F1"
]

display(
    comparison_df[
        comparison_df["Metric"].isin(stage2_metrics)
    ]
)


# --------------------------------------------------
# 6. END-TO-END 29 FEATURE
# --------------------------------------------------

print("\n" + "=" * 70)
print("END-TO-END RESULTS")
print("=" * 70)

e2e_metrics = [
    "End-to-End Accuracy",
    "End-to-End Macro-F1",
    "End-to-End Weighted-F1"
]

display(
    comparison_df[
        comparison_df["Metric"].isin(e2e_metrics)
    ]
)


# --------------------------------------------------
# 7. IMPORTANT NOTE
# --------------------------------------------------

print("\n" + "=" * 70)
print("IMPORTANT")
print("=" * 70)

print(
    "Pipeline 29-feature có đầy đủ End-to-End metrics."
)

print(
    "Pipeline 14-feature chưa điền End-to-End metrics "
    "vì cần chạy lại cùng cách tính để so sánh công bằng."
)

print(
    "Không sử dụng giá trị End-to-End Accuracy = 0.404 "
    "của cell Claude để kết luận ở bước này."
)

print("\nComparison completed.")

In [ ]:
# ==========================================
# EXPORT VERSIONED 29-FEATURE IDS BUNDLE
# ==========================================
import json
import os
import joblib

binary_drop_features = list(drop_features_final)
multiclass_drop_features = list(drop_features_final_mc)
original_features = list(X_train.columns)  # 34 raw predictors in this dataset
required_input_features = [c for c in original_features if c not in binary_drop_features]
multiclass_features = [c for c in original_features if c not in multiclass_drop_features]
categorical_features_deploy = list(candidate_encoder_test.feature_names_in_)
binary_numeric_features = [c for c in required_input_features if c not in categorical_features_deploy]

# Refuse to export if the fitted objects do not match the selected feature set.
assert len(original_features) == 34, f"Expected 34 raw columns, got {len(original_features)}"
assert len(required_input_features) == 29, f"Expected 29 retained inputs, got {len(required_input_features)}"
assert required_input_features == multiclass_features, "Stage feature sets differ"
assert categorical_features_deploy == ["proto", "service", "state"]
assert candidate_model_test.n_features_in_ == 181
assert final_preprocessor_mc.transform(X_attack_full_mc.iloc[:2]).shape[1] == 173
assert list(final_preprocessor_mc.feature_names_in_) == multiclass_features
assert set(candidate_model_test.classes_) == {0, 1}

ids_metadata = {
    "bundle_format": "unsw_nb15_ids_29feature_v1",
    "dataset": "UNSW-NB15",
    "raw_dataset_features": original_features,
    "required_input_features": required_input_features,
    "binary_drop_features": binary_drop_features,
    "multiclass_drop_features": multiclass_drop_features,
    "binary_features": required_input_features,
    "multiclass_features": multiclass_features,
    "categorical_features": categorical_features_deploy,
    "binary_numeric_features": binary_numeric_features,
    "binary_encoded_dimension": int(candidate_model_test.n_features_in_),
    "multiclass_encoded_dimension": 173,
    "binary_classes": [int(v) for v in candidate_model_test.classes_],
    "multiclass_classes": [str(v) for v in final_multiclass_model.classes_],
    "binary_model_type": type(candidate_model_test).__name__,
    "multiclass_model_type": type(final_multiclass_model).__name__,
}

bundle = {
    "binary_model": candidate_model_test,
    "binary_encoder": candidate_encoder_test,
    "multiclass_model": final_multiclass_model,
    "multiclass_preprocessor": final_preprocessor_mc,
    "metadata": ids_metadata,
}

MODEL_DIR = "ids_model_29feature"
os.makedirs(MODEL_DIR, exist_ok=True)
bundle_path = os.path.join(MODEL_DIR, "unsw_nb15_ids_29feature_bundle.joblib")
metadata_path = os.path.join(MODEL_DIR, "metadata.json")
joblib.dump(bundle, bundle_path, compress=3)
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(ids_metadata, f, indent=2, ensure_ascii=False)

print("Exported:", bundle_path)
print("Metadata:", metadata_path)
print("Input columns:", len(required_input_features), "(29 retained from 34 raw predictors)")
print("Stage 1 encoded columns:", ids_metadata["binary_encoded_dimension"])
print("Stage 2 encoded columns:", ids_metadata["multiclass_encoded_dimension"])


In [ ]:
"""Inference helpers for the UNSW-NB15 29-feature, two-stage IDS bundle.

The bundle is exported by the packaging cell in the companion notebook.
Input DataFrames may contain the 29 retained features or all 34 raw dataset
features; the five dropped fields are ignored when present.
"""

from __future__ import annotations

from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd


def load_bundle(path: str | Path) -> dict[str, Any]:
    """Load the versioned joblib bundle created by the Kaggle export cell."""
    bundle = joblib.load(path)
    required = {
        "binary_model",
        "binary_encoder",
        "multiclass_model",
        "multiclass_preprocessor",
        "metadata",
    }
    missing = required.difference(bundle)
    if missing:
        raise ValueError(f"IDS bundle is missing entries: {sorted(missing)}")
    if bundle["metadata"].get("bundle_format") != "unsw_nb15_ids_29feature_v1":
        raise ValueError("Unsupported or missing IDS bundle format marker.")
    return bundle


def predict_ids(input_df: pd.DataFrame, bundle: dict[str, Any]) -> pd.DataFrame:
    """Run Stage 1 (Normal/Attack), then Stage 2 on predicted attacks.

    Returns one row per input sample with binary confidence, attack risk, and
    attack category confidence. Rows predicted Normal have no attack category.
    """
    if not isinstance(input_df, pd.DataFrame):
        raise TypeError("input_df must be a pandas DataFrame")

    metadata = bundle["metadata"]
    required_features = metadata["required_input_features"]
    missing = [name for name in required_features if name not in input_df.columns]
    if missing:
        raise ValueError(f"Missing required 29-feature input columns: {missing}")

    # Select the exact raw feature set and order used by the trained models.
    x = input_df.loc[:, required_features].copy()
    cat_features = metadata["categorical_features"]
    numeric_features = metadata["binary_numeric_features"]

    encoded_cat = bundle["binary_encoder"].transform(x.loc[:, cat_features])
    encoded_binary = np.hstack([x.loc[:, numeric_features].to_numpy(), encoded_cat])
    expected_binary = metadata["binary_encoded_dimension"]
    if encoded_binary.shape[1] != expected_binary:
        raise ValueError(
            f"Stage 1 encoded {encoded_binary.shape[1]} columns; expected {expected_binary}."
        )

    binary_model = bundle["binary_model"]
    binary_pred = binary_model.predict(encoded_binary)
    binary_classes = list(binary_model.classes_)
    if 0 not in binary_classes or 1 not in binary_classes:
        raise ValueError(f"Stage 1 expected classes 0/1, got {binary_classes}.")
    binary_prob = binary_model.predict_proba(encoded_binary)
    normal_col = binary_classes.index(0)
    attack_col = binary_classes.index(1)
    attack_risk = binary_prob[:, attack_col]
    predicted_confidence = binary_prob[
        np.arange(len(binary_pred)),
        [binary_classes.index(label) for label in binary_pred],
    ]

    results: list[dict[str, Any]] = []
    attack_positions = np.flatnonzero(binary_pred == 1)
    attack_labels: dict[int, Any] = {}
    attack_confidences: dict[int, float] = {}

    if len(attack_positions):
        mc_features = metadata["multiclass_features"]
        encoded_mc = bundle["multiclass_preprocessor"].transform(
            x.iloc[attack_positions].loc[:, mc_features]
        )
        expected_mc = metadata["multiclass_encoded_dimension"]
        if encoded_mc.shape[1] != expected_mc:
            raise ValueError(
                f"Stage 2 encoded {encoded_mc.shape[1]} columns; expected {expected_mc}."
            )

        mc_model = bundle["multiclass_model"]
        mc_pred = mc_model.predict(encoded_mc)
        mc_prob = mc_model.predict_proba(encoded_mc)
        mc_classes = list(mc_model.classes_)
        for j, position in enumerate(attack_positions):
            label = mc_pred[j]
            attack_labels[int(position)] = label
            attack_confidences[int(position)] = float(
                mc_prob[j, mc_classes.index(label)]
            )

    for i, label in enumerate(binary_pred):
        is_attack = label == 1
        results.append(
            {
                "prediction": "Attack" if is_attack else "Normal",
                "attack_type": attack_labels.get(i),
                "binary_prediction": int(label),
                "binary_confidence": float(predicted_confidence[i]),
                "attack_probability": float(attack_risk[i]),
                "attack_confidence": attack_confidences.get(i),
            }
        )

    return pd.DataFrame(results, index=input_df.index)


In [ ]:
# ==========================================
# FULL TEST-SET BUNDLE INFERENCE CHECK
# ==========================================
from sklearn.metrics import accuracy_score, f1_score, classification_report

loaded_bundle = load_bundle(bundle_path)
test_input_29 = df_test[loaded_bundle["metadata"]["required_input_features"]].copy()
package_predictions = predict_ids(test_input_29, loaded_bundle)

true_labels = df_test["attack_cat"].astype(str).reset_index(drop=True)
predicted_labels = package_predictions["prediction"].copy()
attack_rows = package_predictions["prediction"].eq("Attack")
predicted_labels.loc[attack_rows] = package_predictions.loc[attack_rows, "attack_type"].astype(str)

print("Samples scored:", len(package_predictions))
print("=== PACKAGED END-TO-END TEST METRICS ===")
print(f"Accuracy    : {accuracy_score(true_labels, predicted_labels):.6f}")
print(f"Macro-F1    : {f1_score(true_labels, predicted_labels, average='macro'):.6f}")
print(f"Weighted-F1 : {f1_score(true_labels, predicted_labels, average='weighted'):.6f}")
print(classification_report(true_labels, predicted_labels, digits=4, zero_division=0))
